# Where in the rule string is the information about kappa?

Registered test (predictions F1-F4 committed before the run, see README). Train: 2185 stationary census rules; test: 162 fresh stationary rules (13-14 letters), disjoint. Models use the rule string only: lookup on the first 8 letters (P8), on the last 8 (S8), regression on composition (C), and P8 plus C. Score: strict held-out R^2 against the training mean.

**Assumptions.** Stationarity filter |g_k| < 0.05 is applied to both train and test (as in the W3c audit); the lookup falls back to shorter keys with at least 3 training rules.

In [1]:
import json
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
d = json.loads((root / "experiments/archive/2026-10-kappa-spread-features/results/features.json").read_text())
print("inputs:", {k: v[:12] for k, v in d["input_sha256"].items()}, "train", d["n_train"], "test", d["n_test"], "overlap", d["overlap"])
for k, v in d["r2"].items():
    lo, hi = d["ci"][k]
    print(f"{k:5s} R^2 {v:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")
print(d["verdicts"])
assert d["overlap"] == 0 and all(d["verdicts"].values())
assert abs(d["r2"]["P8"] - 0.624) < 2e-3 and d["r2"]["S8"] < 0 and d["r2"]["C"] < 0.2

inputs: {'probe_summary.csv': 'c76c7927185b', 'prefix_audit_predictions.csv': '742f0fe1d776'} train 2185 test 162 overlap 0
P8    R^2 +0.624  95% CI [+0.468, +0.733]
S8    R^2 -0.114  95% CI [-0.295, +0.028]
C     R^2 +0.079  95% CI [-0.059, +0.190]
P8+C  R^2 +0.632  95% CI [+0.484, +0.734]
{'F1 P8 within 0.1 of 0.633': True, 'F2 S8 at least 0.2 below P8': True, 'F3 C < 0.2': True, 'F4 P8+C improves on P8 by < 0.05': True}


## What this shows and does not show
- The predictable part of ln kappa is in the first eight letters (R^2 0.62). The last eight letters carry nothing measurable (-0.11; -0.07 in an unregistered leave-one-out control inside the training set), and composition (length, R count, run structure) adds nothing beyond the prefix (+0.008).
- Consistent with an early-evolution / lead-run picture; evidence against kappa being a bulk property of letter composition.
- **Not shown:** which prefix feature matters (the lookup is a family-membership predictor, and families overlap between train and test); what carries the remaining unexplained third of the spread (middle letters and interactions were not tested).

## Next
A registered comparison of structural prefix features (lead length, second-run length) against the 8-letter group mean.